In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import pygraphviz as pgv
import array
import random
import operator

from deap import base
from deap import creator
from deap import tools
from deap import gp

Fix constants. We want a big population since a lot of them will have errors, either mathematical (0/0, inf-inf, log(-a)...) or computational. We want a lot of mutation because trees will be small (at least in the beginning) and mutations can be insignificant in some cases (change equal terminals, change the order of abelian operations...).

In [2]:
MUTATION_RATE = 0.1
CROSSOVER_RATE = 0.7
POPULATION_SIZE = 100
ELITISM = 5
VARIETY = ELITISM

MAX_GENERATIONS = 50
LAG = 5

min_init_depth = 2
max_init_depth = 4
toolbox = base.Toolbox()

random.seed(123)

# Pset


Define safe operations

In [3]:
def safe_div(a,b):
    if b == a:
        return 1
    if b == 0:
        return np.inf
    return a/b

def safe_log(a):
    if a == 0:
        return np.inf
    return np.log(float(abs(a)))

def safe_round(a):
    if abs(a) == np.inf:
        return a
    return round(a)

def safe_pow(a,b):
    # no complex numbers
    if a < 0 and (int(b)-b!=0):
        result = (-a)**b
        return result
    return a**b

Define how constants are generated

In [4]:
def efimeros():
    p = random.random()
    threshold = 0.1 # for constants
    constants = [np.pi,np.e]
    num = len(constants)
    if p<1-threshold:
        return round(random.uniform(-10, 10),1)
    for i in range(num):
        if p<1-threshold*(num-i-1)/num:
            return constants[i]

And add all primitives to pset

In [5]:
pset = gp.PrimitiveSet("MAIN", 1)
pset.renameArguments(ARG0="x")

pset.addPrimitive(operator.add, 2)
pset.addPrimitive(operator.sub, 2)
pset.addPrimitive(operator.mul, 2)
pset.addPrimitive(safe_div, 2)
pset.addPrimitive(safe_pow, 2)
pset.addPrimitive(safe_log, 1)
pset.addPrimitive(safe_round, 1)
pset.addEphemeralConstant("num",efimeros)

# Other deap things

Define evaluation function

In [6]:
def calc_fitness(ind):
    f = toolbox.compile(expr=ind)
    try:
        y_ = np.array([f(x_) for x_ in x])
        fit = np.mean(np.abs(y_ - y)**2)
        if np.isnan(fit) or fit < 0: # sometimes fit is negative for some reason and i don't know why
            ind.fitness.values = tuple([np.inf])
            return tuple([np.inf])

    except Exception:
        ind.fitness.values = tuple([np.inf])
        return tuple([np.inf])
    ind.fitness.values =  tuple([fit])
    return tuple([fit])

We also define how we want to mutate individuals. We want to change them and sometimes shrink them to avoid huge trees

In [7]:
def random_mutation(ind):
    if random.random() < 0.6:
        # at least 1 change, more changes for bigger trees depending on MUTATION_RATE
        changes = int(1+MUTATION_RATE*len(ind)*abs(random.gauss(0,1)))
        for i in range(changes):
            ind = gp.mutNodeReplacement(ind,pset)[0]
    else:
        gp.mutShrink(ind)

toolbox.register("mutate", random_mutation)

Now register all needed operations in the toolbox

In [8]:
toolbox.register("mate", gp.cxOnePoint)

toolbox.register("compile", gp.compile, pset=pset)
creator.create("FitnessMin", base.Fitness, weights=(-1.0,)) # we want the minimum error
creator.create("Individual", gp.PrimitiveTree,fitness=creator.FitnessMin, prim_set=pset) # "type" of individual
toolbox.register("expr", gp.genHalfAndHalf,pset=pset, min_=min_init_depth, max_=max_init_depth) # how we generate
toolbox.register("individual", tools.initIterate,container=creator.Individual,generator=toolbox.expr) # individual generator
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("select", tools.selTournament, tournsize=5)
# toolbox.register("select", tools.selBest, k=int(POPULATION_SIZE/10))


# Evolution

create initial population. Ignore errors from bad people

In [9]:
def create_population():
    hof = tools.HallOfFame(10)
    popu = toolbox.population(n=POPULATION_SIZE)
    fit = list(map(calc_fitness, popu)) # fit isn't really needed as a variable, but if we don't save the interpreter omits the map
    hof.update(popu)
    return hof,popu

We evolve it up to a number of generation or if we stop improving for a few generations. The strategy is the following:
- Select some of the original population through tournament
- Mutate and crossover this to create offspring
- Keep some of the best from the previous generation, the offspring and some new and random individuals to encourage diversity

Every generation updates the hall of fame. It is close to redundant, but because it avoids repeating the best and keeps some best results possible lost by elitism we still do it.

In [10]:
def evolve(hof,popu):
    best=[]
    g = 0
    for i in range(LAG):
        g += 1
        print(f"Gen {g}: ", end='')
        offspring = toolbox.select(popu, POPULATION_SIZE-ELITISM-VARIETY)
        # Clone the selected individuals
        offspring = list(map(toolbox.clone, offspring))

        # mutate and crossover
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CROSSOVER_RATE:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            if random.random() < MUTATION_RATE:
                toolbox.mutate(mutant)
                del mutant.fitness.values
        popu = tools.selBest(popu,ELITISM)+offspring+toolbox.population(n=VARIETY)

        fit = list(map(calc_fitness, popu))
        best.append(min(fit)[0])
        hof.update(popu)

        print(best[-1])

    while g < MAX_GENERATIONS and best[-1] != best[-LAG]:
        g += 1
        print(f"Gen {g}: ", end='')
        offspring = toolbox.select(popu, POPULATION_SIZE-ELITISM-VARIETY)
        offspring = list(map(toolbox.clone, offspring))

        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CROSSOVER_RATE:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            if random.random() < MUTATION_RATE:
                toolbox.mutate(mutant)
                del mutant.fitness.values
        popu = tools.selBest(popu,ELITISM)+offspring+toolbox.population(n=VARIETY)

        fit = list(map(calc_fitness, popu))
        best.append(min(fit)[0])
        hof.update(popu)

        print(best[-1])
    return best

# Run and analyze

In [11]:
data = pd.read_csv('pi.csv', header=None)

x = data.iloc[1:,0].values
y = data.iloc[1:,1].values

hof1, popu1 = create_population()
best1 = evolve(hof1,popu1)

FileNotFoundError: [Errno 2] No such file or directory: 'pi.csv'

In [ ]:
for i in hof1:
  print(i.fitness,i)
f = toolbox.compile(hof1[0])

data = pd.read_csv('pi.csv', header=None)
plt.plot(x, y)
plt.plot(x, [f(i) for i in x])
plt.show()

plt.semilogy(best1)
plt.show()

In [ ]:
data = pd.read_csv('eul.csv', header=None)

x = data.iloc[1:,0].values
y = data.iloc[1:,1].values

hof2, popu2 = create_population()
best2 = evolve(hof2,popu2)

In [ ]:
for i in hof2:
  print(i.fitness,i)
f = toolbox.compile(hof1[0])

data = pd.read_csv('pi.csv', header=None)
plt.plot(x, y)
plt.plot(x, [f(i) for i in x])
plt.show()

plt.semilogy(best2)
plt.show()